# Missing Metadata Bar Chart

Purpose: Create bar chart that shows how much data has metadata (collection date, state) missing from SRA (Andersen Lab)

Three categories:
* Collection date and state
* No collection date, yes state
* Missing metadata

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 

In [2]:
# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"

# Day we're updating data
update_date = "05-20-2025"

os.chdir(downloads)

In [3]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# Looking for B3.13 only

# Get genotype from genoflu_results.tsv

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
metadata = metadata[metadata["Genotype"] == "B3.13"]

print(metadata)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
2     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
7     SRR28752453        WGS      144.71   37055919  PRJNA1102327   
9     SRR28752455        WGS      147.62   63694864  PRJNA1102327   
...           ...        ...         ...        ...           ...   
8862  SRR33370187        WGS      149.06  266567663  PRJNA1207547   
8863  SRR33370188        WGS      149.12  252084447  PRJNA1207547   
8864  SRR33370189        WGS      147.86  126955445  PRJNA1207547   
8865  SRR33370190        WGS      148.58  124032966  PRJNA1207547   
8866  SRR33370191        WGS      148.75  261906821  PRJNA1207547   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
2     SAMN41019236          Viral  24547283   USDA-NVSL            2024  ...   
3     SAMN4

In [4]:
# Get specific geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

print(genbank_mapping)

metadata_genbank = metadata.merge(genbank_mapping, on=["Run"], how="left") # Include data without states

print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank) # Maybe there is no state information since 3/18/2025?

                    seg_file  \
0      SRR28752446_HA_cns.fa   
8      SRR28752447_HA_cns.fa   
16     SRR28752448_HA_cns.fa   
24     SRR28752449_HA_cns.fa   
32     SRR28752450_HA_cns.fa   
...                      ...   
40738  SRR33124773_HA_cns.fa   
40746  SRR33124774_HA_cns.fa   
40754  SRR33124775_HA_cns.fa   
40762  SRR33124776_HA_cns.fa   
40770  SRR33124777_HA_cns.fa   

                                            seg_seq_name      sra_run seg  \
0      Consensus_SRR28752446_HA_cns_threshold_0.5_qua...  SRR28752446  HA   
8      Consensus_SRR28752447_HA_cns_threshold_0.5_qua...  SRR28752447  HA   
16     Consensus_SRR28752448_HA_cns_threshold_0.5_qua...  SRR28752448  HA   
24     Consensus_SRR28752449_HA_cns_threshold_0.5_qua...  SRR28752449  HA   
32     Consensus_SRR28752450_HA_cns_threshold_0.5_qua...  SRR28752450  HA   
...                                                  ...          ...  ..   
40738  Consensus_SRR33124773_HA_cns_threshold_0.5_qua...  SRR33124773  HA   

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,NaN,B3.13,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4.0,A/cattle/Texas/24-009108-004/2024,Texas
1,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,NaN,B3.13,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4.0,A/cattle/Texas/24-009108-003/2024,Texas
2,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,NaN,B3.13,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4.0,A/cattle/Texas/24-009108-002/2024,Texas
3,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,NaN,B3.13,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4.0,A/cattle/Texas/24-009088-001/2024,Texas
4,SRR28752455,WGS,147.62,63694864,PRJNA1102327,SAMN41019229,Viral,19920569,USDA-NVSL,2024,...,NaN,B3.13,SRR28752455_HA_cns.fa,Consensus_SRR28752455_HA_cns_threshold_0.5_qua...,SRR28752455,HA,PP752661.1,4.0,A/cattle/Texas/24-009029-001/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4324,SRR33370187,WGS,149.06,266567663,PRJNA1207547,SAMN48201686,Viral,96596674,USDA-NVSL,2025,...,NaN,B3.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4325,SRR33370188,WGS,149.12,252084447,PRJNA1207547,SAMN48201685,Viral,91406525,USDA-NVSL,2025,...,NaN,B3.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4326,SRR33370189,WGS,147.86,126955445,PRJNA1207547,SAMN48201684,Viral,52294003,USDA-NVSL,2025,...,NaN,B3.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4327,SRR33370190,WGS,148.58,124032966,PRJNA1207547,SAMN48201654,Viral,50995262,USDA-NVSL,2025,...,NaN,B3.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def search_collection_date(biosample, metadata_genbank):

    print(biosample)

    try:
    
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        # Get Biosample ID from search_url
        output = requests.get(search_url)
        xml = output.content
        root = ET.fromstring(xml)
        sample_id = root.find("./IdList/Id").text

        biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
        
        # Get Nucleotide ID from biosample_url
        output = requests.get(biosample_url)
        xml = output.content
        root = ET.fromstring(xml)
        query_key = root.find(".//QueryKey").text
        web_env = root.find(".//WebEnv").text

        nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        print(nucleotide_url)

        output = requests.get(nucleotide_url) 
        xml = output.content
        root = ET.fromstring(xml)

        # Grab collection date at the end of the sub name
        collection_date = root.find(".//SubName").text.split("|")[-1]

        # Grab geo_location as well
        geo_location = root.find(".//SubName").text.split("/")[2]

        # If there's a state associated, re-format
        geo_location = geo_location.replace(": ", "-")

        print(collection_date)

        # Avoid spamming the server
        time.sleep(2)

        return geo_location + "|" + collection_date
    
    except:
        print("Unable to find collection date.")

        if len(metadata_genbank[metadata_genbank["BioSample"] == biosample]["Collection_Date"]) > 0: # If a year exists
            collection_date = metadata_genbank[metadata_genbank["BioSample"] == biosample]["Collection_Date"].values[0]
        else:
            collection_date = float('nan') 

        # Avoid spamming the server
        time.sleep(1)

        return collection_date

In [ ]:
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank_" + update_date + ".csv")

SAMN41019236
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_682cbceeb4e8a6ee3905c3b5&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
SAMN41019235
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_682cbcf287150cb1ce00ef29&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
SAMN41019234
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_682cbcf56ca85cae3d03fbf3&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
SAMN41019231
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_682cbcf7dca83d6f31004eb7&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
13-Mar-2024
SAMN41019229
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_682cbcf9ab0df90343067dd1&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
21

In [25]:
# Comment the above cell out if using this cell

os.chdir(originals + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_" + update_date + ".csv")
metadata_genbank["Collection_Date_State"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: x.split("|")[-1] if pd.notna(x) and x.split("|")[-1] != x.split("|")[0] else np.nan)
metadata_genbank["No_Collection_Date"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: x.split("|")[0] if pd.notna(x) and x.split("|")[0] != x.split("|")[-1] else np.nan)
# "Neither" category is denoted by nan in collection_date_specific

print(metadata_genbank["Collection_Date_Specific"].apply(pd.isna).sum()) # 33 "neither"'s

print(metadata_genbank)

33
      Unnamed: 0          Run Assay Type  AvgSpotLen      Bases    BioProject  \
0              0  SRR28752448        WGS      250.30   75035343  PRJNA1102327   
1              1  SRR28752449        WGS      146.61   59363690  PRJNA1102327   
2              2  SRR28752450        WGS      251.31  119232569  PRJNA1102327   
3              3  SRR28752453        WGS      144.71   37055919  PRJNA1102327   
4              4  SRR28752455        WGS      147.62   63694864  PRJNA1102327   
...          ...          ...        ...         ...        ...           ...   
4324        4324  SRR33370187        WGS      149.06  266567663  PRJNA1207547   
4325        4325  SRR33370188        WGS      149.12  252084447  PRJNA1207547   
4326        4326  SRR33370189        WGS      147.86  126955445  PRJNA1207547   
4327        4327  SRR33370190        WGS      148.58  124032966  PRJNA1207547   
4328        4328  SRR33370191        WGS      148.75  261906821  PRJNA1207547   

         BioSample BioSa

In [ ]:
# print(metadata_genbank["ReleaseDate"]) # release date by month is x-axis 
# y-axis: number of sequences released in that month
# categories: state and collection date, state no collection date, neither 

# print(metadata_genbank)

0       2024-04-20 18:20:39
1       2024-04-20 18:20:39
2       2024-04-20 18:20:39
3       2024-04-20 18:20:39
4       2024-04-20 18:20:39
               ...         
4324    2025-05-02 01:11:03
4325    2025-05-02 01:11:02
4326    2025-05-02 01:11:02
4327    2025-05-02 01:10:28
4328    2025-05-02 01:10:27
Name: ReleaseDate, Length: 4329, dtype: object
      Unnamed: 0          Run Assay Type  AvgSpotLen      Bases    BioProject  \
0              0  SRR28752448        WGS      250.30   75035343  PRJNA1102327   
1              1  SRR28752449        WGS      146.61   59363690  PRJNA1102327   
2              2  SRR28752450        WGS      251.31  119232569  PRJNA1102327   
3              3  SRR28752453        WGS      144.71   37055919  PRJNA1102327   
4              4  SRR28752455        WGS      147.62   63694864  PRJNA1102327   
...          ...          ...        ...         ...        ...           ...   
4324        4324  SRR33370187        WGS      149.06  266567663  PRJNA1207547 